# LeetCode 3715 — Sum of Perfect Square Ancestors bằng Python

- Link: https://leetcode.com/problems/sum-of-perfect-square-ancestors/
- Độ khó: **Hard**
- Chủ đề: Số học (**phần không chính phương** / square-free kernel, sàng ước nguyên tố nhỏ nhất), DFS trên cây với **bộ đếm theo đường đi**, Euler tour + Fenwick

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def sumOfAncestors(self, n: int, edges: List[List[int]], nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Chứng minh: `a·b` là số chính phương ⇔ `core(a) = core(b)`, với `core(x)` = tích các số nguyên tố có **số mũ lẻ** trong `x`.
2. Tính `core` cho mọi số `≤ 10^5` bằng sàng **ước nguyên tố nhỏ nhất** (SPF).
3. Đếm tổ tiên cùng `core` bằng DFS duy trì **bộ đếm trên đường đi gốc → nút** (thêm khi vào, bớt khi ra), không đệ quy.
4. Cách khác: **Euler tour** + Fenwick đếm cặp (tổ tiên, hậu duệ) trong cùng nhóm `core`.
5. Kiểm chứng bằng duyệt tổ tiên trực tiếp với `isqrt`.

## 2. Đề bài & ví dụ

Cây gốc `0`, mỗi nút có `nums[i] > 0`. `t_i` = số tổ tiên `a` của `i` (không tính `i`) sao cho `nums[i]·nums[a]` là số chính phương. Trả về `Σ_{i=1}^{n−1} t_i`.

**Ví dụ 1:** `n=3, edges=[[0,1],[1,2]], nums=[2,8,2]` → `3`.

**Ví dụ 2:** `n=3, edges=[[0,1],[0,2]], nums=[1,2,4]` → `1`.

**Ví dụ 3:** `n=4, edges=[[0,1],[0,2],[1,3]], nums=[1,2,9,4]` → `2`.

**Ràng buộc:** `n ≤ 10^5`, `nums[i] ≤ 10^5`.

In [1]:
from typing import List
from math import isqrt
from collections import defaultdict
import random, time

examples = [
    (3, [[0, 1], [1, 2]], [2, 8, 2], 3),
    (3, [[0, 1], [0, 2]], [1, 2, 4], 1),
    (4, [[0, 1], [0, 2], [1, 3]], [1, 2, 9, 4], 2),
]
for n, e, a, exp in examples:
    print(n, e, a, "-> kỳ vọng", exp)

3 [[0, 1], [1, 2]] [2, 8, 2] -> kỳ vọng 3
3 [[0, 1], [0, 2]] [1, 2, 4] -> kỳ vọng 1
4 [[0, 1], [0, 2], [1, 3]] [1, 2, 9, 4] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Phần không chính phương

Viết `x = core(x) · y²` với `core(x)` không có ước chính phương `> 1`. Khi đó `a·b = core(a)·core(b)·(y_a y_b)²` là số chính phương ⇔ `core(a)·core(b)` là số chính phương ⇔ `core(a) = core(b)` (hai số square-free có tích chính phương thì bằng nhau: mỗi số nguyên tố phải xuất hiện ở cả hai hoặc không ở số nào).

In [2]:
def core_slow(x):
    c = 1; p = 2
    while p * p <= x:
        e = 0
        while x % p == 0: x //= p; e += 1
        if e % 2: c *= p
        p += 1
    return c * x

for a, b in [(2, 8), (2, 2), (12, 3), (12, 27), (5, 20), (6, 10)]:
    sq = isqrt(a * b) ** 2 == a * b
    print(f"{a}·{b}={a*b}: chính phương={sq}, core={core_slow(a)},{core_slow(b)} -> bằng nhau={core_slow(a) == core_slow(b)}")

2·8=16: chính phương=True, core=2,2 -> bằng nhau=True
2·2=4: chính phương=True, core=2,2 -> bằng nhau=True
12·3=36: chính phương=True, core=3,3 -> bằng nhau=True
12·27=324: chính phương=True, core=3,3 -> bằng nhau=True
5·20=100: chính phương=True, core=5,5 -> bằng nhau=True
6·10=60: chính phương=False, core=6,10 -> bằng nhau=False


### 3.2. Sàng ước nguyên tố nhỏ nhất (SPF)

`spf[x]` = ước nguyên tố nhỏ nhất của `x`. Có `spf`, phân tích `x` trong `O(log x)`. Còn có thể tính `core` cho mọi `x` bằng quy hoạch: `core[x] = core[x / p]` chia hoặc nhân `p` tuỳ `p` đã có trong `core[x/p]` chưa.

In [3]:
def core_table(N):
    spf = list(range(N + 1))
    for i in range(2, isqrt(N) + 1):
        if spf[i] == i:
            for j in range(i * i, N + 1, i):
                if spf[j] == j: spf[j] = i
    core = [0] * (N + 1); core[1] = 1
    for x in range(2, N + 1):
        p = spf[x]; c = core[x // p]
        core[x] = c // p if c % p == 0 else c * p          # lật số mũ của p
    return core

CORE = core_table(10**5)
assert all(CORE[x] == core_slow(x) for x in range(1, 3000))
print([CORE[x] for x in range(1, 21)])

[1, 2, 3, 1, 5, 6, 7, 2, 1, 10, 11, 3, 13, 14, 15, 1, 17, 2, 19, 5]


### 3.3. Bộ đếm trên đường đi

Khi DFS từ gốc, tập các nút "đang mở" (đã vào, chưa ra) đúng bằng đường đi gốc → nút hiện tại. Duy trì `cnt[c]` = số nút đang mở có `core = c`: tăng khi vào, giảm khi ra. Tại nút `i`, trước khi tăng, `cnt[core(i)]` chính là `t_i`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`t_i` = số tổ tiên có cùng `core` với `i`. Đáp án = số cặp (tổ tiên, hậu duệ) có cùng `core`.

Ví dụ 1: `core = [2, 2, 2]` (vì `8 = 2·2²`). Cây là đường thẳng `0 − 1 − 2` ⇒ cặp `(0,1), (0,2), (1,2)` ⇒ `3`. ✔

## 5. Cách 1 — Brute Force: duyệt tổ tiên, kiểm tra `isqrt`

In [4]:
def parents(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    par = [-1] * n; order = [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v in adj[u]:
            if not seen[v]: seen[v] = True; par[v] = u; order.append(v)
    return adj, par, order

def brute(n, edges, nums):
    _, par, _ = parents(n, edges); total = 0
    for i in range(1, n):
        a = par[i]
        while a != -1:
            p = nums[i] * nums[a]
            if isqrt(p) ** 2 == p: total += 1
            a = par[a]
    return total

for n, e, a, exp in examples:
    print(brute(n, e, a), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
2 kỳ vọng 2


**Nút thắt:** `O(Σ độ sâu)` — tới `O(n²) = 5·10^9` với cây đường thẳng.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `core` bằng sàng (3.2).
2. DFS không đệ quy với sự kiện "vào" và "ra": vào nút `u` ⇒ `ans += cnt[core(u)]`, `cnt[core(u)] += 1`; ra ⇒ `cnt[core(u)] −= 1`.

Tổng `O(N_max + n)`.

## 7. Các cách tiếp theo

- **Cách 2:** duyệt tổ tiên nhưng so `core` (nhanh hơn `isqrt`, vẫn `O(n · độ sâu)`).
- **Cách 3:** DFS + bộ đếm trên đường đi — `O(n)` (lời giải nộp).
- **Cách 4:** Euler tour + Fenwick theo từng nhóm `core`: cặp (tổ tiên `a`, hậu duệ `v`) ⇔ `tin[a] < tin[v] ≤ tout[a]` — `O(n log n)`.

### 7.1. Cách 2 — Duyệt tổ tiên so `core`

In [5]:
def walk_core(n, edges, nums):
    _, par, _ = parents(n, edges); c = [CORE[x] for x in nums]; total = 0
    for i in range(1, n):
        a = par[i]
        while a != -1:
            total += c[a] == c[i]; a = par[a]
    return total

for n, e, a, exp in examples:
    print(walk_core(n, e, a), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
2 kỳ vọng 2


### 7.2. Cách 3 — DFS + bộ đếm đường đi (lời giải chính)

In [6]:
def dfs_counter(n, edges, nums):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    c = [CORE[x] for x in nums]
    cnt = defaultdict(int); ans = 0
    stack = [(0, -1, False)]
    while stack:
        u, p, leaving = stack.pop()
        if leaving:
            cnt[c[u]] -= 1; continue
        ans += cnt[c[u]]; cnt[c[u]] += 1
        stack.append((u, p, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, False))
    return ans

for n, e, a, exp in examples:
    print(dfs_counter(n, e, a), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
2 kỳ vọng 2


### 7.3. Cách 4 — Euler tour + Fenwick theo nhóm

In [7]:
def euler_fenwick(n, edges, nums):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    tin = [0] * n; tout = [0] * n; t = 0; stack = [(0, -1, False)]
    while stack:
        u, p, leaving = stack.pop()
        if leaving: tout[u] = t - 1; continue
        tin[u] = t; t += 1; stack.append((u, p, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, False))
    groups = defaultdict(list)
    for u in range(n): groups[CORE[nums[u]]].append(u)
    bit = [0] * (n + 1); ans = 0
    def add(i, d):
        i += 1
        while i <= n: bit[i] += d; i += i & -i
    def pre(i):
        i += 1; s = 0
        while i > 0: s += bit[i]; i -= i & -i
        return s
    for g in groups.values():
        for u in g: add(tin[u], 1)
        for u in g: ans += pre(tout[u]) - pre(tin[u])          # hậu duệ thật sự cùng nhóm
        for u in g: add(tin[u], -1)
    return ans

for n, e, a, exp in examples:
    print(euler_fenwick(n, e, a), "kỳ vọng", exp)

3 kỳ vọng 3
1 kỳ vọng 1
2 kỳ vọng 2


## 8. TRACE từng bước

In [8]:
def trace(n, edges, nums):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    c = [CORE[x] for x in nums]
    print("nums =", nums, "| core =", c)
    cnt = defaultdict(int); ans = 0; stack = [(0, -1, False)]
    while stack:
        u, p, leaving = stack.pop()
        if leaving:
            cnt[c[u]] -= 1; print(f"  ra {u}: cnt = {dict(cnt)}"); continue
        add = cnt[c[u]]; ans += add; cnt[c[u]] += 1
        print(f"  vào {u} (core {c[u]}): t_{u} = {add}, ans = {ans}, cnt = {dict(cnt)}")
        stack.append((u, p, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, False))
    return ans

trace(4, [[0, 1], [0, 2], [1, 3]], [1, 2, 9, 4])

nums = [1, 2, 9, 4] | core = [1, 2, 1, 1]
  vào 0 (core 1): t_0 = 0, ans = 0, cnt = {1: 1}
  vào 2 (core 1): t_2 = 1, ans = 1, cnt = {1: 2}
  ra 2: cnt = {1: 1}
  vào 1 (core 2): t_1 = 0, ans = 1, cnt = {1: 1, 2: 1}
  vào 3 (core 1): t_3 = 1, ans = 2, cnt = {1: 2, 2: 1}
  ra 3: cnt = {1: 1, 2: 1}
  ra 1: cnt = {1: 1, 2: 0}
  ra 0: cnt = {1: 0, 2: 0}


2

In [9]:
trace(6, [[0, 1], [1, 2], [0, 3], [3, 4], [4, 5]], [3, 12, 27, 5, 20, 3])
print("brute:", brute(6, [[0, 1], [1, 2], [0, 3], [3, 4], [4, 5]], [3, 12, 27, 5, 20, 3]))

nums = [3, 12, 27, 5, 20, 3] | core = [3, 3, 3, 5, 5, 3]
  vào 0 (core 3): t_0 = 0, ans = 0, cnt = {3: 1}
  vào 3 (core 5): t_3 = 0, ans = 0, cnt = {3: 1, 5: 1}
  vào 4 (core 5): t_4 = 1, ans = 1, cnt = {3: 1, 5: 2}
  vào 5 (core 3): t_5 = 1, ans = 2, cnt = {3: 2, 5: 2}
  ra 5: cnt = {3: 1, 5: 2}
  ra 4: cnt = {3: 1, 5: 1}
  ra 3: cnt = {3: 1, 5: 0}
  vào 1 (core 3): t_1 = 1, ans = 3, cnt = {3: 2, 5: 0}
  vào 2 (core 3): t_2 = 2, ans = 5, cnt = {3: 3, 5: 0}
  ra 2: cnt = {3: 2, 5: 0}
  ra 1: cnt = {3: 1, 5: 0}
  ra 0: cnt = {3: 0, 5: 0}
brute: 5


## 9. Code nộp LeetCode

In [10]:
from math import isqrt

class Solution:
    def sumOfAncestors(self, n: int, edges: List[List[int]], nums: List[int]) -> int:
        N = max(nums)
        spf = list(range(N + 1))
        for i in range(2, isqrt(N) + 1):
            if spf[i] == i:
                for j in range(i * i, N + 1, i):
                    if spf[j] == j:
                        spf[j] = i
        core = [0] * (N + 1)
        core[1] = 1
        for x in range(2, N + 1):
            p = spf[x]
            c = core[x // p]
            core[x] = c // p if c % p == 0 else c * p
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)
        key = [core[x] for x in nums]
        cnt = {}
        ans = 0
        stack = [(0, -1, False)]
        while stack:
            u, p, leaving = stack.pop()
            k = key[u]
            if leaving:
                cnt[k] -= 1
                continue
            c = cnt.get(k, 0)
            ans += c
            cnt[k] = c + 1
            stack.append((u, p, True))
            for v in adj[u]:
                if v != p:
                    stack.append((v, u, False))
        return ans

sol = Solution()
for n, e, a, exp in examples:
    got = sol.sumOfAncestors(n, e, a)
    print(got, "✔" if got == exp else "✘")

3 ✔
1 ✔
2 ✔


## 10. Vì sao thuật toán đúng

1. **Đặc trưng số học** (3.1): `nums[i]·nums[a]` chính phương ⇔ `core(nums[i]) = core(nums[a])`.
2. **`core` bằng sàng:** `x = p · (x/p)` với `p = spf(x)`; nhân thêm `p` lật tính chẵn lẻ số mũ của `p` ⇒ nếu `p | core(x/p)` (mũ lẻ) thì thành chẵn ⇒ chia `p`; ngược lại nhân `p`. Quy nạp theo `x`.
3. **Bất biến DFS:** khi pop sự kiện "vào `u`", `cnt` đếm đúng các nút trên đường gốc → cha(`u`) (các nút đã vào nhưng chưa ra — sự kiện "ra" của nút `w` được đẩy trước các con của `w` nên chỉ xử lý sau khi xong cây con của `w`). Vậy `cnt[core(u)]` = `t_u`.
4. **Euler tour:** `a` là tổ tiên thật sự của `v` ⇔ `tin[a] < tin[v] ≤ tout[a]`; đếm theo từng nhóm `core` bằng BIT rồi xoá để dùng lại.

## 11. Phân tích Time / Space Complexity

Với `V = max(nums)`:

| Cách | Time | Space |
|---|---|---|
| 1. Tổ tiên + `isqrt` | `O(n · depth)` | `O(n)` |
| 2. Tổ tiên + `core` | `O(V log log V + n · depth)` | `O(V + n)` |
| 3. DFS + bộ đếm | `O(V log log V + n)` | `O(V + n)` |
| 4. Euler + Fenwick | `O(V log log V + n log n)` | `O(V + n)` |

## 12. Bộ test

In [11]:
METHODS = [brute, walk_core, dfs_counter, euler_fenwick, lambda n, e, a: Solution().sumOfAncestors(n, e, a)]
for n, e, a, exp in examples:
    assert all(f(n, e, a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().sumOfAncestors(1, [], [7]) == 0
assert Solution().sumOfAncestors(2, [[1, 0]], [4, 9]) == 1          # cạnh cho ngược chiều, core đều = 1
assert Solution().sumOfAncestors(3, [[0, 1], [1, 2]], [1, 1, 1]) == 3
assert Solution().sumOfAncestors(3, [[0, 1], [0, 2]], [2, 3, 5]) == 0
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 cây ngẫu nhiên
def rand_tree(rng, n):
    e = [[i, rng.randrange(i)] for i in range(1, n)]
    for x in e:
        if rng.random() < 0.5: x.reverse()
    return e

rng = random.Random(3715)
pool = [1, 2, 3, 4, 5, 6, 8, 9, 12, 18, 20, 27, 32, 45, 50, 72, 98]
for _ in range(1500):
    n = rng.randint(1, 12); e = rand_tree(rng, n); a = [rng.choice(pool) for _ in range(n)]
    b = brute(n, e, a)
    assert all(f(n, e, a) == b for f in METHODS[1:]), (n, e, a)
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn: đường thẳng sâu 10^5 và cây ngẫu nhiên
rng = random.Random(1)
for name, e in [("đường thẳng", [[i, i + 1] for i in range(10**5 - 1)]), ("ngẫu nhiên", rand_tree(rng, 10**5))]:
    a = [rng.randint(1, 10**5) for _ in range(10**5)]
    t0 = time.time(); x = Solution().sumOfAncestors(10**5, e, a); t1 = time.time()
    print(f"{name}: đáp án {x}, thời gian {t1 - t0:.2f}s, khớp Euler: {x == euler_fenwick(10**5, e, a)}")

đường thẳng: đáp án 358722, thời gian 0.12s, khớp Euler: True


ngẫu nhiên: đáp án 113, thời gian 0.20s, khớp Euler: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh "cả hai đều là số chính phương"

`8·2 = 16` chính phương dù 8 và 2 đều không chính phương.

In [15]:
def wrong_both_square(n, edges, nums):
    _, par, _ = parents(n, edges); is_sq = lambda x: isqrt(x) ** 2 == x; total = 0
    for i in range(1, n):
        a = par[i]
        while a != -1:
            total += is_sq(nums[i]) and is_sq(nums[a]); a = par[a]       # SAI
    return total

print("sai:", wrong_both_square(3, [[0, 1], [1, 2]], [2, 8, 2]), "| đúng:", brute(3, [[0, 1], [1, 2]], [2, 8, 2]))

sai: 0 | đúng: 3


### Lỗi 2 — Dùng "tập ước nguyên tố" thay vì phần square-free

`12 = 2²·3` và `3`: cùng tập nguyên tố có mũ lẻ `{3}` ⇒ đúng. Nhưng `12` và `6` có cùng **tập ước nguyên tố** `{2, 3}` mà `72` không chính phương.

In [16]:
def prime_set(x):
    s = set(); p = 2
    while p * p <= x:
        while x % p == 0: s.add(p); x //= p
        p += 1
    if x > 1: s.add(x)
    return frozenset(s)

print("12, 6: cùng tập ước nguyên tố:", prime_set(12) == prime_set(6), "| 72 chính phương?", isqrt(72) ** 2 == 72,
      "| core:", core_slow(12), core_slow(6))

12, 6: cùng tập ước nguyên tố: True | 72 chính phương? False | core: 3 6


### Lỗi 3 — Quên giảm bộ đếm khi rời nút (đếm cả "anh em họ")

In [17]:
def wrong_no_decrement(n, edges, nums):
    adj, par, order = parents(n, edges); c = [CORE[x] for x in nums]
    cnt = defaultdict(int); ans = 0
    stack = [0]; seen = {0}
    while stack:                                                         # SAI: không có sự kiện "ra"
        u = stack.pop(); ans += cnt[c[u]]; cnt[c[u]] += 1
        for v in adj[u]:
            if v not in seen: seen.add(v); stack.append(v)
    return ans

print("sai:", wrong_no_decrement(3, [[0, 1], [0, 2]], [1, 4, 9]), "| đúng:", brute(3, [[0, 1], [0, 2]], [1, 4, 9]))

sai: 3 | đúng: 2


### Lỗi 4 — DFS đệ quy với cây sâu `10^5`

Python mặc định giới hạn đệ quy 1000 ⇒ `RecursionError` trên cây đường thẳng. Dùng ngăn xếp tường minh với cờ "vào/ra" như lời giải.

In [18]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit(), "< độ sâu tối đa 10^5")

giới hạn đệ quy mặc định: 1000 < độ sâu tối đa 10^5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tích là số chính phương" ⇒ **phần không chính phương** bằng nhau.
- "Đếm tổ tiên thoả điều kiện bằng nhau" ⇒ DFS với bộ đếm trên đường đi (thêm khi vào, bớt khi ra).
- Nhiều truy vấn phân tích thừa số với số `≤ 10^5–10^7` ⇒ sàng SPF.
- Cặp (tổ tiên, hậu duệ) ⇔ khoảng Euler lồng nhau ⇒ Fenwick/sweep.

## 15. Biến thể của bài

1. **Đếm cặp bất kỳ** (không cần quan hệ tổ tiên) có tích chính phương: `Σ_c C(cnt_c, 2)`.
2. **Tích là lập phương**: so "vectơ số mũ mod 3" — chuẩn hoá thành `core3(x)` và cần `core3(a) · core3(b)` lập phương ⇔ `core3(b) = bù(core3(a))`.
3. **Chỉ tổ tiên trong `k` mức**: bộ đếm trên đường đi kết hợp cửa sổ độ sâu (loại nút ở độ sâu `< depth − k`).
4. **Truy vấn trên đường đi bất kỳ u–v**: dùng LCA + bộ đếm theo tiền tố đường đi.

In [19]:
# Biến thể 3: chỉ tính tổ tiên cách tối đa k mức
def within_k(n, edges, nums, k):
    from bisect import bisect_left
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    c = [CORE[x] for x in nums]
    depths = defaultdict(list)          # với mỗi core: độ sâu (tăng dần) của các nút cùng core trên đường đi
    ans = 0; stack = [(0, -1, 0, False)]
    while stack:
        u, p, d, leaving = stack.pop()
        if leaving:
            depths[c[u]].pop(); continue
        lst = depths[c[u]]
        ans += len(lst) - bisect_left(lst, d - k)        # tổ tiên cùng core có độ sâu >= d - k
        lst.append(d); stack.append((u, p, d, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, d + 1, False))
    return ans

def within_k_brute(n, edges, nums, k):
    _, par, _ = parents(n, edges); total = 0
    for i in range(1, n):
        a = par[i]; d = 1
        while a != -1 and d <= k:
            total += CORE[nums[a]] == CORE[nums[i]]; a = par[a]; d += 1
    return total

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 12); e = rand_tree(rng, n); a = [rng.choice([1, 2, 4, 8, 3, 12]) for _ in range(n)]; k = rng.randint(1, 4)
    assert within_k(n, e, a, k) == within_k_brute(n, e, a, k), (n, e, a, k)
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Tổ tiên + `isqrt` | Định nghĩa | `O(n · depth)` | Kiểm chứng |
| 2. Tổ tiên + `core` | Rút gọn số học | `O(n · depth)` | Cây nông |
| 3. DFS + bộ đếm | Đường đi gốc → nút | `O(n)` | **Lời giải chuẩn** |
| 4. Euler + Fenwick | Khoảng lồng nhau | `O(n log n)` | Khi cần offline/tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Đường thẳng `0−1−2−3`, `nums = [2, 18, 8, 3]`.

**Bài 2 (tính tay).** Sao tâm 0 với 4 lá, `nums = [5, 20, 45, 6, 80]`.

**Bài 3 (điền TODO).** Viết `count_square_pairs(nums)` = số cặp `i < j` bất kỳ có `nums[i]·nums[j]` chính phương.

**Bài 4 (tự viết lại).** Viết `core` cho một số `x ≤ 10^12` bằng chia thử tới `x^{1/3}` rồi xử lý phần dư (gợi ý: phần dư là 1, `p`, `p²` hoặc `pq`).

**Bài 5 (mini challenge).** Trả về mảng `t` (từng `t_i`) thay vì tổng.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_square_pairs(nums):
    # TODO
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def per_node(n, edges, nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `core = [2, 2, 2, 3]`. (2) Cặp tổ tiên cùng core: `(0,1), (0,2), (1,2)`. (3) `3`.

**Bài 2.** (1) `core = [5, 5, 5, 6, 5]`. (2) Lá `1, 2, 4` khớp gốc. (3) `3`.

**Bài 3.** (1) Nhóm theo `core`. (2) `Σ C(c, 2)`. (3) Kiểm chứng với `isqrt`.

**Bài 4.** (1) Chia thử mọi `p ≤ x^{1/3}`. (2) Phần dư `r` không có ước `≤ x^{1/3}` ⇒ `r ∈ {1, p, p², pq}`. (3) `r` chính phương ⇒ góp 1, ngược lại góp `r`.

**Bài 5.** (1) Lưu `t[u] = cnt[core(u)]` khi vào. (2) Trả mảng. (3) Tổng khớp lời giải.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().sumOfAncestors(4, [[0, 1], [1, 2], [2, 3]], [2, 18, 8, 3]) == 3
assert Solution().sumOfAncestors(5, [[0, i] for i in range(1, 5)], [5, 20, 45, 6, 80]) == 3
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def count_square_pairs_sol(nums):
    g = defaultdict(int)
    for x in nums: g[CORE[x]] += 1
    return sum(c * (c - 1) // 2 for c in g.values())

def core_big(x):
    c = 1; p = 2
    while p * p * p <= x:
        e = 0
        while x % p == 0: x //= p; e += 1
        if e % 2: c *= p
        p += 1
    r = isqrt(x)
    return c if r * r == x else c * x

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(1, 200) for _ in range(rng.randint(1, 10))]
    exp = sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if isqrt(a[i] * a[j]) ** 2 == a[i] * a[j])
    assert count_square_pairs_sol(a) == exp
for x in [1, 12, 72, 10**12, 999983 * 999979, 999983 ** 2 * 7, 2 * 3 * 5 * 7 * 11 * 13 * 17 * 19 * 23 * 29]:
    assert core_big(x) == core_slow(x) if x < 10**7 else True
assert core_big(999983 ** 2 * 7) == 7 and core_big(999983 * 999979) == 999983 * 999979
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def per_node_sol(n, edges, nums):
    adj = [[] for _ in range(n)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    c = [CORE[x] for x in nums]; cnt = defaultdict(int); t = [0] * n
    stack = [(0, -1, False)]
    while stack:
        u, p, leaving = stack.pop()
        if leaving: cnt[c[u]] -= 1; continue
        t[u] = cnt[c[u]]; cnt[c[u]] += 1; stack.append((u, p, True))
        for v in adj[u]:
            if v != p: stack.append((v, u, False))
    return t

print(per_node_sol(3, [[0, 1], [1, 2]], [2, 8, 2]), per_node_sol(4, [[0, 1], [0, 2], [1, 3]], [1, 2, 9, 4]))
assert per_node_sol(3, [[0, 1], [1, 2]], [2, 8, 2]) == [0, 1, 2]

[0, 1, 2] [0, 0, 1, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
spf sàng; core[x] = core[x//p] // p nếu p | core[x//p] ngược lại * p
stack = [(0, -1, False)]
while stack:
    u, p, leaving = stack.pop()
    if leaving: cnt[core[u]] -= 1; continue
    ans += cnt[core[u]]; cnt[core[u]] += 1
    stack.append((u, p, True)); đẩy các con
```

**Checklist:**

- [ ] Dùng **phần không chính phương** (không phải tập ước nguyên tố)?
- [ ] Đếm trước, tăng sau (không tự đếm chính mình)?
- [ ] Giảm bộ đếm khi rời nút?
- [ ] Không đệ quy sâu?
- [ ] Cạnh cho vô hướng, gốc là 0?